# SynBioCrow 2.3 — Galaxy Development Evaluation V3\nOffline evaluation only. Verifies the sealed prediction hash and reconstructs two ensemble graphs: strict stereochemical identity and connectivity identity with stereochemistry removed before graph construction. No backend chemistry is rerun.\n

In [ ]:
import os,sys,subprocess,shutil,json,zipfile,hashlib
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_galaxy_eval_v3")
SEALED=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_development_sealed")
NORM=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_literature_normalized")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_development_evaluation_v3")
OUT.mkdir(parents=True,exist_ok=True)

def run(cmd,*,cwd=None,timeout=1200):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if p.returncode: raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)

pred=SEALED/"sealed_predictions.json"
bench=NORM/"galaxy_literature_benchmark_normalized.json"
manifest=json.loads((SEALED/"sealed_manifest.json").read_text())
actual=hashlib.sha256(pred.read_bytes()).hexdigest()
assert actual==manifest["predictions_sha256"],(actual,manifest["predictions_sha256"])
print("SEALED PREDICTIONS VERIFIED",actual)

run([sys.executable,str(REPO/"scripts/evaluate_galaxy_development_v3_2_3.py"),
     "--predictions",str(pred),"--benchmark",str(bench),"--output-dir",str(OUT)],cwd=REPO)

bundle=OUT/"SynBioCrow_2_3_GALAXY_DEVELOPMENT_EVALUATION_V3.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*.json")): z.write(p,p.name)
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
